In [ ]:
import os
import numpy as np
from datetime import datetime
from numpy.lib.recfunctions import structured_to_unstructured as s2u
from sgt2npy import load_ndarray_from_sgt
from scipy.signal import medfilt, spectrogram, detrend
import matplotlib.pyplot as plt

from librosa import amplitude_to_db, power_to_db
from librosa.display import specshow
from librosa.feature import melspectrogram, mfcc

# import matplotlib
# matplotlib.use('tkagg')

In [ ]:
datetime.fromtimestamp(1732533416).strftime('%H:%M:%S')

In [ ]:
ABL_Z_K = 2.750813464e-02
ABR_Z_K = 2.931076739e-02
FFT = 512
SR = 2000

def load_baks(path):
    data, info = load_ndarray_from_sgt(path)
    data = s2u(data[['Second', 'Index', 'chABL_Z', 'chABR_Z']]).astype(np.float64)
    data[:, 2] *= ABL_Z_K
    data[:, 3] *= ABR_Z_K
    return data, info

def load_mway(path):
    data, info = load_ndarray_from_sgt(path)
    data = s2u(data[['$Second', '$Index', 'pSpeed']])
    return data, info

def merge_by_time(data_aks, data_way):
    """не совпадают секунды в файлах aks и mway"""
    time_1 = data_aks[:, 0]
    time_2 = data_way[:, 0]
    l_b = np.max((time_1[0], time_2[0]))
    r_b = np.min((time_1[-1], time_2[-1]))
    data_res_aks = data_aks[np.where(data_aks[:, 0] == l_b+1)[0][0]: np.where(data_aks[:, 0] == r_b-1)[0][0]]
    data_res_way = data_way[np.where(data_way[:, 0] == l_b+1)[0][0]: np.where(data_way[:, 0] == r_b-1)[0][0]]
    return data_res_aks, data_res_way


In [ ]:
# src1 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_110123TBAKS_00.sgt"  # 2 тестовые
# src2 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_110123TMWAY_00.sgt"

# src1 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_135222TBAKS_02.sgt" # не робит
# src2 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_135222TMWAY_02.sgt"

src1 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_135222TBAKS_01.sgt"  # 1
src2 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_135222TMWAY_01.sgt"
src3 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_135222TMWAY_02.sgt"

data_aks, info_aks = load_baks(src1)
data_way, info_way = load_mway(src2)
data_way2, info_way = load_mway(src3)
data_way = np.vstack((data_way, data_way2))

# data_aks, data_way = merge_by_time_1(data_aks, data_way)

# # подрезать половину долгой стоянки в начале
# data_aks = data_aks[int(0.2e7):, :]
# data_way = data_way[int(0.2e7):, :]

# fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 6))
# ax[0].plot(data_aks[:, 2])
# ax[0].set_title('Данные буксового акселерометра')
# ax[1].plot(data_way[:, 2])
# ax[1].set_title('Скорость, м/с по датчику ДПС')
# ax[1].set_xlabel('Отсчеты. Частота дискретизации: 3000 Гц')
# plt.show()
# print(data_aks[:, 2].shape, data_way[:, 2].shape)

In [ ]:
# прилепить еще данных

src4 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_110123TBAKS_00.sgt"  # 2 тестовые
src5 = r"\\10.1.1.69\data2\ДКИ-513\WayDB\01\2024_11\1\513\20241125_110123TMWAY_00.sgt"

data_aks_2, info_aks = load_baks(src4)
data_way_2, info_way = load_mway(src5)

data_aks_2, data_way_2 = merge_by_time(data_aks_2, data_way_2)

data_aks = np.vstack((data_aks, data_aks_2))
data_way = np.vstack((data_way, data_way_2))

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 6))
ax[0].plot(data_aks[:, 2])
ax[0].set_title('Данные буксового акселерометра')
ax[1].plot(data_way[:, 2])
ax[1].set_title('Скорость, м/с по датчику ДПС')
ax[1].set_xlabel('Отсчеты. Частота дискретизации: 3000 Гц')
plt.show()
print(data_aks[:, 2].shape, data_way[:, 2].shape)

data_aks = data_aks[:, 2]
data_way = data_way[:, 2]

In [ ]:
np.save('data\\raw_data_aks.npy', data_aks)
np.save('data\\raw_data_way.npy', data_way)

In [ ]:
# перемешка

# ind = np.arange(data_aks[:, 2].shape[0])
# ind_perm = np.random.permutation(ind)
# data_perm = data_aks[ind_perm, 2]
# np.save('data\\X_test_perm.npy', data_perm)
# y_perm = data_way[ind_perm, 2]
# np.save('data\\y_test_perm.npy', y_perm)

In [ ]:
# фильтр. режет высокие частоты

flt = np.ones(2501, np.float64)/2501
aks_hf = np.convolve(data_aks, flt, mode='valid') # mode=valid немного режет размер (у результата всё равно будет интерполяция), mode=same по краям имеет выбросы.

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(data_aks)
ax[1].plot(aks_hf)


In [ ]:
np.save('data\\raw_data_aks_f.npy', aks_hf)
np.save('data\\raw_data_way.npy', data_way)

In [ ]:
# Спектрограмма

data_aks_f = medfilt(data_aks, kernel_size=5)  # фильтруется шум

f, t, Sxx = spectrogram(data_aks_f, fs=SR, nperseg=FFT, noverlap=32, detrend='linear', mode='psd')

print(Sxx.shape)

Sxx_db = power_to_db(Sxx)
# Sxx_db_nf = power_to_db(Sxx_nf)
fig, ax = plt.subplots( figsize=(10, 3))
# ax[1].set_title('После фильтра высоких частот')
plt.tight_layout()
# specshow(Sxx_db_nf, y_axis='hz', x_axis='s', ax=ax[0])
specshow(Sxx_db, y_axis='hz', x_axis='s', ax=ax)

### Mel + MFCC

In [ ]:
mel_spec = melspectrogram(S=Sxx, sr=SR, n_fft=FFT, n_mels=128, power=2, center=False) #fmin=0, fmax=200
plt.figure(figsize=(10, 3))
mel_spec_db = power_to_db(mel_spec)
specshow(mel_spec_db, sr=SR, x_axis='time', y_axis='mel')
print(mel_spec.shape)


In [ ]:
mfcc_spec = mfcc(
    S=power_to_db(mel_spec),
    sr=SR,
    n_mfcc=128,
    dct_type=2,
    norm="ortho",
)

mfcc_spec_dtr = detrend(mfcc_spec, type='linear')
plt.figure(figsize=(10, 3))
specshow(power_to_db(mfcc_spec_dtr), y_axis='hz', sr=SR)
print(mfcc_spec.shape)


In [ ]:
plt.plot(np.unique(mfcc_spec_dtr))

In [ ]:
ind = np.arange(mfcc_spec.shape[-1])
rnd_ind = np.random.permutation(ind)
rnd_mfcc = mfcc_spec_dtr[:, rnd_ind]
np.save('perm_ind.npy', rnd_ind)
specshow(power_to_db(rnd_mfcc), y_axis='hz', sr=SR)

In [ ]:
np.save('data\\X_train_new.npy', mfcc_spec_dtr)

In [ ]:
# shape of data_way --> mfcc.shape[1]
data_way_interp = np.interp(np.linspace(0, 1, mfcc_spec.shape[1]), np.linspace(0, 1, data_way.shape[0]), data_way)
plt.figure(figsize=(12, 3))
plt.plot(data_way_interp)

In [ ]:
# data_way_interp_rnd = data_way_interp[rnd_ind]

np.save('data\\y_train_new.npy', data_way_interp)

### New data

In [ ]:
import os
import numpy as np
from sgt2npy import load_ndarray_from_p3 as load
# from matplotlib import pyplot as plt

# tmp_fn = r"\\10.1.1.69\data2\Velaro\Июнь_2020\20200601_083856\DBX2_%s.p3"
# tmp_fn_mway = r"\\10.1.1.69\data2\Velaro\Июнь_2020\20200601_083856\MWAY_%s.p3"

# tmp_fn = r"\\10.1.1.69\data2\Velaro\Июнь_2020\20200531_150855\DBX2_%s.p3"
# tmp_fn_mway = r"\\10.1.1.69\data2\Velaro\Июнь_2020\20200531_150855\MWAY_%s.p3"
# r"\\10.1.1.69\data2\Velaro\Июнь_2020\20200531_150855"

tmp_fn = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180920_205115\DBX1_%s.p3"
tmp_fn_mway = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180920_205115\MWAY_%s.p3"

# tmp_fn = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180920_151739\DBX1_%s.p3"
# tmp_fn_mway = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180920_151739\MWAY_%s.p3"

# tmp_fn = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180923_131717\DBX2_%s.p3"
# tmp_fn_mway = r"\\10.1.1.69\data2\Velaro\Сентябрь_2018_Ускорения+КУХ\measurements\20180923_131717\MWAY_%s.p3"




# =============================================================================
# n = 0
#
# fn = tmp_fn % f'{n:03d}'
# data, info = load(fn)
# time = data['Second'] + data['Index']/info.consts.get('frequency').value
# dt = np.dtype([('Time', np.float64)])
# =============================================================================


def load_file(fn: str, channels: list) -> np.recarray:
    data, info = load(fn)
    dt = [('Time', np.float64)]
    for ch in channels:
        dt.append((ch, data.dtype[ch].type))
    dt = np.dtype(dt)
    result = np.zeros(len(data), dtype=dt)
    result['Time'] = data['Second'] + data['Index']/info.consts.get('frequency').value
    result[channels] = data[channels]
    return result

def part_name(n: int, tmp: str=tmp_fn) -> str:
    return tmp % f'{n:03d}'

part = 0
data = None

while os.path.isfile(part_name(part, tmp_fn)):
    part_data = load_file(part_name(part, tmp_fn), ['aks_x', 'aks_y', 'aks_z'])
    if data is None:
        data = part_data
    else:
        data = np.hstack([data, part_data])
    part += 1

part = 0
mway = None

while os.path.isfile(part_name(part, tmp_fn_mway)):
    part_data = load_file(part_name(part, tmp_fn_mway), ['speed',])
    if mway is None:
        mway = part_data
    else:
        mway = np.hstack([mway, part_data])
    part += 1

# Стыковка границ aks и mway по времени
print(data.shape, mway.shape)
lb = np.max((data['Time'][0], mway['Time'][0])) + 1
rb = np.min((data['Time'][-1], mway['Time'][-1])) - 1
ind_a = np.where((lb < data['Time']) & (data['Time'] < rb))
ind_w = np.where((lb < mway['Time']) & (mway['Time'] < rb))
data = data[ind_a]
mway = mway[ind_w]
print(data.shape, mway.shape)

ch = 'aks_x'
aks = data[ch].astype(np.float32)
aks -= np.average(aks)
aks *= 9.81/236

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(data['Time'], aks)
ax[1].plot(mway['Time'], mway['speed'])

mway_ipl = np.interp(np.linspace(0, 1, aks.shape[0]), np.linspace(0, 1, mway['speed'].shape[0]), mway['speed'])
print('final', aks.shape, mway_ipl.shape)


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(data['Time'], aks)
ax[1].plot(mway['Time'], mway['speed'])
ax[1].set_ylabel('скорость м/с')

In [ ]:
np.save(r'data\20180920_205115_DBX1.npy', aks)
np.save(r'data\20180920_205115_mway.npy', mway_ipl)

### Tuda siuda

In [ ]:
# Fixing random state for reproducibility
np.random.seed(19680801)

dt = 0.0005  # шаг дискретизации (расстояние между отсчетами)
t = np.arange(0.0, 20.5, dt)
s1 = np.sin(2 * np.pi * 30 * t)
s2 = 2 * np.sin(2 * np.pi * 80 * t)

# create a transient "chirp"
s2[t <= 10] = s2[12 <= t] = 0

# add some noise into the mix
nse = 0.01 * np.random.random(size=len(t))

x_nl = s1 + s2
x = s1 + s2 + nse  # the signal
NFFT = 1024  # the length of the windowing segments
Fs = 1/dt  # the sampling frequency

fig, (ax1, ax2) = plt.subplots(nrows=2, sharex=True)
ax1.plot(t, x)
ax1.set_ylabel('Signal')

Pxx, freqs, bins, im = ax2.specgram(x, NFFT=NFFT, Fs=Fs)
# The `specgram` method returns 4 objects. They are:
# - Pxx: the periodogram
# - freqs: the frequency vector
# - bins: the centers of the time bins
# - im: the .image.AxesImage instance representing the data in the plot
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Frequency (Hz)')
# ax2.set_xlim(0, 20)

plt.show()

In [ ]:
def get_sin(A, freq, t, phi):
    return A * np.sin(2 * np.pi * freq * t + phi)

t = np.arange(0, 6, 0.01)
A = 0.5
f1 = 0.5
f2 = 1
# phi1 = 1.5
phi1 = 0
phi2 = 0

s1 = get_sin(A, f1, t, phi1)
s2 =  get_sin(A, f2, t, phi2)

S = s1+s2

plt.plot(t, s1)
# plt.plot(t, s1, t, s2, t, S)


In [ ]:
# Параметры сигнала
t = np.arange(0, 6, 0.01)
A1 = 0.5
A2 = 1
f1 = 0.5
f2 = 1
phi1 = 0.2
phi2 = 0

# Функция для генерации синусоиды
def get_sin(A, f, t, phi):
    return A * np.sin(2 * np.pi * f * t + phi)

# Генерируем сигналы
s1 = get_sin(A1, f1, t, phi1)
s2 = get_sin(A2, f2, t, phi2)
S = s1 + s2

# Визуализация исходных сигналов
plt.figure(figsize=(12, 4))
plt.plot(t, s1, label='s1 (0.5 Hz)', alpha=0.7)
plt.plot(t, s2, label='s2 (1 Hz)', alpha=0.7)
plt.plot(t, S, label='S = s1 + s2', linewidth=2)
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.title('Исходные сигналы')
plt.legend()
plt.grid(True)
plt.show()

# Выполняем FFT
N = len(S)  # Количество точек
S_fft = np.fft.fft(S)  # Прямое преобразование Фурье
S_fft_mag = np.abs(S_fft) / N  # Амплитуда (нормированная)
S_fft_phase = np.angle(S_fft)  # Фаза

# Частотная ось
fs = 1 / (t[1] - t[0])  # Частота дискретизации
freqs = np.fft.fftfreq(N, 1/fs)  # Частоты для FFT

# Оставляем только положительные частоты (для реального сигнала)
half_N = N // 2
freqs_positive = freqs[:half_N]
S_fft_mag_positive = S_fft_mag[:half_N] * 2  # Умножаем на 2 для одностороннего спектра
S_fft_phase_positive = S_fft_phase[:half_N]

# Визуализация спектра амплитуд
plt.figure(figsize=(12, 4))
plt.stem(freqs_positive, S_fft_mag_positive, basefmt=" ")
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда')
plt.title('Спектр амплитуд (FFT)')
plt.xlim(0, 2)  # Ограничиваем диапазон для наглядности
plt.grid(True)
plt.show()

# Визуализация спектра фаз
plt.figure(figsize=(12, 4))
plt.stem(freqs_positive, S_fft_phase_positive, basefmt=" ")
plt.xlabel('Частота (Гц)')
plt.ylabel('Фаза (рад)')
plt.title('Спектр фаз (FFT)')
plt.xlim(0, 2)
plt.grid(True)
plt.show()

In [ ]:
from configparser import ConfigParser

cfg = ConfigParser()
cfg.read('cfg.ini')

cfg['Routes']['img_folder']
cfg.get('Routes', 'img_folder')
cfg.get('Routes', 'mask_folder')
print(list(cfg['Routes'].keys()))


In [ ]:
from scipy.signal import savgol_filter, medfilt


signal_st = np.zeros((10000))
signal_st[signal_st.shape[0] // 5 * 2: signal_st.shape[0] // 5 * 3] = 1


plt.plot(signal_st)

### Old

In [ ]:
smoothed_data = medfilt(data_aks[:, 2], kernel_size=5)
fig, ax = plt.subplots()
Pxx, freqs, bins, im = ax.specgram(smoothed_data, NFFT=FFT, Fs=info_aks.header.frequency, cmap='inferno', noverlap=32, scale='dB',
                                   detrend='linear')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
plt.show()
print(Pxx.shape, freqs.shape, bins.shape)
#
# freqs диапозон частот от 0 до 1500 (част дискр 257)
# bins диапозон времени от 0 до 4990 (част дискр 31191)
print('Pxx', Pxx[0], 'freqs', freqs[0], freqs[-1], 'bins', bins[0], bins[-1])

In [ ]:
Pxx_db = power_to_db(Pxx)
specshow(Pxx_db)

In [ ]:
# Отображение отдельно Pxx

Pxx_db = 10 * np.log10(Pxx) # power_to_db(S)

plt.figure(figsize=(10, 3))
plt.imshow(Pxx_db,
           aspect='auto',           # Сохраняем пропорции
           origin='lower',        # Начало координат внизу
           extent=[bins[0], bins[-1], freqs[0], freqs[-1]],
           cmap='viridis')         # Цветовая схема

plt.xlabel('Время (с)')
plt.ylabel('Частота (Гц)')
plt.title('Спектральная плотность мощности (Pxx)')
plt.colorbar(label='Мощность (дБ)')
plt.show()

In [ ]:
# Mel spectrogramm
from librosa.filters import mel
from sklearn.preprocessing import MinMaxScaler
from librosa import amplitude_to_db
from librosa.display import specshow


sample_rate = 3000
n_mels = 128

mel_matrix = mel(sr=sample_rate, n_fft=FFT, n_mels=n_mels)
mel_matrix = MinMaxScaler().fit_transform(mel_matrix.T).T

mel_spec = np.dot(mel_matrix, Pxx)
# mel_spec_db = amplitude_to_db(abs(mel_spec))
mel_spec_db = power_to_db(abs(mel_spec))
plt.figure(figsize=(14, 4))
plt.title(f'Мел-спектрограмма {n_mels} фильтров')
specshow(mel_spec_db, sr=sample_rate, x_axis='time', y_axis='mel')
plt.show()


In [ ]:

mfcc_librosa = mfcc(
    S=power_to_db(mel_spec),
    n_mfcc=128,
    dct_type=2,
    norm="ortho",
)

specshow(power_to_db(mfcc_librosa))


In [ ]:
plot_spectrogram(mfcc_librosa_dtr, title="MFCC (librosa)")